Multi-Agent System for Hospital Appointment Scheduling

Introduction

Hospital appointment scheduling is a complex operational task that requires coordinating patients, physicians, medical specialties, available time slots, patient preferences, and scheduling constraints. In real healthcare environments, scheduling is also dynamic because appointments may be cancelled, new requests may arrive, physicians may become temporarily unavailable, and doctors may change their schedules.

This project investigates whether a Multi-Agent System (MAS) can improve hospital appointment scheduling compared with a Single-Agent baseline under the same information, constraints, and scheduling environment.

The project follows a controlled experimental approach. First, a Single-Agent scheduling system is developed as the baseline. The baseline centrally handles appointment requests by identifying feasible appointments, evaluating candidate assignments according to patient preferences, and selecting the best available option. The resulting schedules are then validated against the defined constraints and evaluated using several performance metrics.

The experiments are conducted using synthetic hospital data, including physicians, patients, appointment requests, and time slots. The first experiment uses a relatively simple dataset to establish the basic scheduling behaviour of the baseline system. The second experiment increases the problem complexity by introducing a larger dataset, multiple appointment requests per patient, different appointment durations, physician capacity constraints, maximum waiting periods, and dynamic events.

The dynamic experiment evaluates how the Single-Agent system responds when the scheduling environment changes after the initial schedule has been created. These changes include patient cancellations, physician unavailability, new appointment requests, and physician schedule changes.

The same datasets, constraints, events, and evaluation metrics will later be used to evaluate the proposed Multi-Agent System. This controlled setup allows the performance of the Single-Agent and Multi-Agent approaches to be compared under equivalent conditions.

The main research question addressed by this project is:

Does a Multi-Agent System improve hospital appointment scheduling performance compared with a Single-Agent approach as the scheduling environment becomes more complex and dynamic?

The main evaluation criteria include:

• Successful assignment rate
• Constraint violations
• Patient preference satisfaction
• Date and time satisfaction
• Average waiting or date deviation
• Dynamic event success rate
• Appointment recovery after disruptions

The results of the Single-Agent experiments therefore provide the baseline performance against which the Multi-Agent System will be compared.

Experiment 1 — Easy Dataset: Single-Agent Baseline

This experiment establishes the initial Single-Agent scheduling baseline using a controlled and relatively simple synthetic hospital dataset.

The purpose of this experiment is to verify the basic scheduling workflow before introducing larger datasets and dynamic changes. The system generates physicians, patients, appointment requests, and available time slots, and then uses a centralized scheduling agent to assign suitable appointments.

The resulting schedule is validated against the defined scheduling constraints and evaluated using patient satisfaction and assignment performance metrics.

Doctors Dataset

This section creates a synthetic dataset representing physicians in a simplified hospital environment.

Each physician is assigned a medical specialty, years of experience, working hours, and a maximum number of appointments that can be handled per day. These attributes represent the basic resources and constraints required for appointment scheduling.

The physician dataset will later be shared between the Single-Agent and Multi-Agent systems so that both approaches operate under the same healthcare environment and can be compared fairly.

In [1]:
import pandas as pd
import random

random.seed(42)

SPECIALTIES = [
    "Cardiology",
    "Dermatology",
    "Neurology",
    "Pediatrics",
    "Orthopedics",
    "General Medicine",
    "Internal Medicine",
    "ENT",
    "Ophthalmology",
    "Obstetrics and Gynecology"
]

def generate_doctors(num_doctors):
    doctors = []

    for i in range(1, num_doctors + 1):
        specialty = SPECIALTIES[(i - 1) % len(SPECIALTIES)]

        doctors.append({
            "doctor_id": f"D{i:03d}",
            "specialty": specialty,
            "experience_years": random.randint(3, 20),
            "work_start": "09:00",
            "work_end": "17:00",
            "max_daily_appointments": random.randint(1, 8)
        })

    return pd.DataFrame(doctors)

doctors_df = generate_doctors(20)

print("===== DOCTORS DATASET =====")
print("Number of doctors:", len(doctors_df))
print()
print(doctors_df)

===== DOCTORS DATASET =====
Number of doctors: 20

   doctor_id                  specialty  ...  work_end max_daily_appointments
0       D001                 Cardiology  ...     17:00                      1
1       D002                Dermatology  ...     17:00                      4
2       D003                  Neurology  ...     17:00                      3
3       D004                 Pediatrics  ...     17:00                      2
4       D005                Orthopedics  ...     17:00                      1
5       D006           General Medicine  ...     17:00                      2
6       D007          Internal Medicine  ...     17:00                      4
7       D008                        ENT  ...     17:00                      1
8       D009              Ophthalmology  ...     17:00                      4
9       D010  Obstetrics and Gynecology  ...     17:00                      7
10      D011                 Cardiology  ...     17:00                      8
11      D012 

Doctors Dataset Result

The generated dataset contains 20 physicians distributed across 10 medical specialties. Each physician has defined working hours and a daily appointment capacity.

This dataset provides the available physician resources that the scheduling system can use when matching patients with appropriate doctors.

The use of synthetic but structured physician data allows the scheduling process to be controlled while maintaining the main characteristics required for a hospital appointment scheduling problem.

Patients Dataset

This section generates the synthetic patient dataset used by the scheduling system.

Each patient is assigned a required medical specialty and may have a preferred physician. Patients also specify a preferred appointment date and time period. These preferences are important because the scheduling system should not only produce valid appointments, but should also attempt to satisfy patient preferences whenever possible.

The same patient dataset will be used by both scheduling approaches to maintain a fair experimental comparison.

In [2]:
# Part 2 — Patients Dataset

from datetime import datetime, timedelta

PATIENT_COUNT = 100

PREFERRED_TIME_OPTIONS = [
    "Morning",
    "Afternoon",
    "No Preference"
]

def generate_patients(num_patients, doctors_df, start_date):
    patients = []

    for i in range(1, num_patients + 1):
        patient_id = f"P{i:03d}"

        required_specialty = random.choice(SPECIALTIES)

        specialty_doctors = doctors_df[
            doctors_df["specialty"] == required_specialty
        ]["doctor_id"].tolist()

        if random.random() < 0.40:
            preferred_doctor_id = random.choice(specialty_doctors)
        else:
            preferred_doctor_id = None

        preferred_date = start_date + timedelta(
            days=random.randint(0, 6)
        )

        preferred_time = random.choice(PREFERRED_TIME_OPTIONS)

        patients.append({
            "patient_id": patient_id,
            "required_specialty": required_specialty,
            "preferred_doctor_id": preferred_doctor_id,
            "preferred_date": preferred_date.strftime("%Y-%m-%d"),
            "preferred_time": preferred_time
        })

    return pd.DataFrame(patients)


START_DATE = datetime(2026, 9, 14)

patients_df = generate_patients(
    PATIENT_COUNT,
    doctors_df,
    START_DATE
)

print("===== PATIENTS DATASET =====")
print("Number of patients:", len(patients_df))
print()

print(patients_df.head(10))

===== PATIENTS DATASET =====
Number of patients: 100

  patient_id         required_specialty  ... preferred_date preferred_time
0       P001              Ophthalmology  ...     2026-09-14  No Preference
1       P002                Orthopedics  ...     2026-09-18      Afternoon
2       P003  Obstetrics and Gynecology  ...     2026-09-14  No Preference
3       P004                 Pediatrics  ...     2026-09-14        Morning
4       P005                Dermatology  ...     2026-09-19      Afternoon
5       P006                  Neurology  ...     2026-09-19      Afternoon
6       P007                Dermatology  ...     2026-09-15  No Preference
7       P008                 Pediatrics  ...     2026-09-16  No Preference
8       P009              Ophthalmology  ...     2026-09-20        Morning
9       P010                 Pediatrics  ...     2026-09-20      Afternoon

[10 rows x 5 columns]


Patients Dataset Result

The generated dataset contains 100 patients. Each patient has a required medical specialty and scheduling preferences, including a preferred date and time. Some patients also specify a preferred physician, while others allow the system to select any suitable physician.

This dataset provides the patient-side requirements that the scheduling algorithm must satisfy while considering physician availability and scheduling constraints.

Appointment Requests

This section converts the patient information into standardized appointment requests.

Each request represents a patient’s need to schedule an appointment and contains the information required by the scheduling system, including the patient identifier, required specialty, preferred physician, preferred date, and preferred time.

Using standardized appointment requests provides a common input format that can later be supplied to both the Single-Agent and Multi-Agent systems.

In [3]:
# Part 3 — Appointment Requests

def generate_appointment_requests(patients_df):
    requests = []

    for _, patient in patients_df.iterrows():

        request_id = f"R{len(requests) + 1:03d}"

        requests.append({
            "request_id": request_id,
            "patient_id": patient["patient_id"],
            "required_specialty": patient["required_specialty"],
            "preferred_doctor_id": patient["preferred_doctor_id"],
            "preferred_date": patient["preferred_date"],
            "preferred_time": patient["preferred_time"]
        })

    return pd.DataFrame(requests)


appointment_requests_df = generate_appointment_requests(
    patients_df
)

print("===== APPOINTMENT REQUESTS =====")
print("Number of appointment requests:", len(appointment_requests_df))
print()

print(appointment_requests_df.head(10))

===== APPOINTMENT REQUESTS =====
Number of appointment requests: 100

  request_id patient_id  ... preferred_date preferred_time
0       R001       P001  ...     2026-09-14  No Preference
1       R002       P002  ...     2026-09-18      Afternoon
2       R003       P003  ...     2026-09-14  No Preference
3       R004       P004  ...     2026-09-14        Morning
4       R005       P005  ...     2026-09-19      Afternoon
5       R006       P006  ...     2026-09-19      Afternoon
6       R007       P007  ...     2026-09-15  No Preference
7       R008       P008  ...     2026-09-16  No Preference
8       R009       P009  ...     2026-09-20        Morning
9       R010       P010  ...     2026-09-20      Afternoon

[10 rows x 6 columns]


Appointment Requests Result

A total of 100 appointment requests were generated, corresponding to the 100 patients in the Easy experiment.

Each request provides the scheduling requirements that the Single-Agent scheduler must evaluate. This ensures that the scheduler works with explicit and structured inputs rather than directly modifying the patient dataset.

Time Slots and Doctor Availability

This section creates the time environment used for appointment scheduling.

Each physician’s working day is divided into fixed one-hour time slots over the scheduling period. Every slot contains the physician, date, start time, end time, and current availability status.

These slots represent the hospital’s available scheduling resources. The scheduler can only assign an appointment when a suitable slot is available and satisfies the patient’s and physician’s constraints

In [4]:
# Part 4 — Time Slots & Doctor Availability

WORK_START_TIME = 9
WORK_END_TIME = 17

SLOT_DURATION_MINUTES = 60

SCHEDULING_DAYS = 6


def generate_time_slots(doctors_df, start_date):
    slots = []

    for _, doctor in doctors_df.iterrows():

        doctor_id = doctor["doctor_id"]

        for day_offset in range(SCHEDULING_DAYS):

            current_date = start_date + timedelta(days=day_offset)

            current_time = datetime.combine(
                current_date,
                datetime.min.time()
            ).replace(hour=WORK_START_TIME)

            end_time = datetime.combine(
                current_date,
                datetime.min.time()
            ).replace(hour=WORK_END_TIME)

            while current_time < end_time:

                slot_end_time = current_time + timedelta(
                    minutes=SLOT_DURATION_MINUTES
                )

                slots.append({
                    "slot_id": f"S{len(slots) + 1:04d}",
                    "doctor_id": doctor_id,
                    "date": current_date.strftime("%Y-%m-%d"),
                    "start_time": current_time.strftime("%H:%M"),
                    "end_time": slot_end_time.strftime("%H:%M"),
                    "status": "Available"
                })

                current_time = slot_end_time

    return pd.DataFrame(slots)


slots_df = generate_time_slots(
    doctors_df,
    START_DATE
)


print("===== TIME SLOTS DATASET =====")
print("Number of doctors:", len(doctors_df))
print("Scheduling days:", SCHEDULING_DAYS)
print("Slot duration:", SLOT_DURATION_MINUTES, "minutes")
print("Working hours:", f"{WORK_START_TIME}:00 - {WORK_END_TIME}:00")
print("Total slots:", len(slots_df))
print()

print(slots_df.head(10))

===== TIME SLOTS DATASET =====
Number of doctors: 20
Scheduling days: 6
Slot duration: 60 minutes
Working hours: 9:00 - 17:00
Total slots: 960

  slot_id doctor_id        date start_time end_time     status
0   S0001      D001  2026-09-14      09:00    10:00  Available
1   S0002      D001  2026-09-14      10:00    11:00  Available
2   S0003      D001  2026-09-14      11:00    12:00  Available
3   S0004      D001  2026-09-14      12:00    13:00  Available
4   S0005      D001  2026-09-14      13:00    14:00  Available
5   S0006      D001  2026-09-14      14:00    15:00  Available
6   S0007      D001  2026-09-14      15:00    16:00  Available
7   S0008      D001  2026-09-14      16:00    17:00  Available
8   S0009      D001  2026-09-15      09:00    10:00  Available
9   S0010      D001  2026-09-15      10:00    11:00  Available


Time Slots Result

The experiment contains 20 physicians over 6 scheduling days, with one-hour appointment slots between 09:00 and 17:00.

This produces a total of 960 time slots. All slots are initially available and can be selected by the scheduling system when they satisfy the required constraints.

The generated slots therefore form the basic temporal resource used by the Single-Agent scheduler.

Single-Agent Scheduling Logic

This section implements the core scheduling logic of the Single-Agent baseline.

The Single-Agent centrally performs the main scheduling tasks. For each appointment request, it identifies feasible physician-slot combinations, removes assignments that violate constraints, calculates a score for the remaining candidates, and selects the highest-scoring valid assignment.

The scoring mechanism considers patient preferences such as preferred physician, preferred date, and preferred time. Importantly, feasibility is checked before scoring so that an invalid assignment cannot be selected simply because it receives a high preference score.

This centralized workflow provides the baseline architecture that will later be compared with the Multi-Agent approach.

In [5]:
# Part 5 — Single-Agent Scheduling Logic

from datetime import datetime


# ---------------------------------------------------------
# 1. Check whether the selected time matches the preference
# ---------------------------------------------------------

def matches_time_preference(start_time, preferred_time):

    hour = int(start_time.split(":")[0])

    if preferred_time == "Morning":
        return hour < 12

    elif preferred_time == "Afternoon":
        return hour >= 12

    else:
        return True


# ---------------------------------------------------------
# 2. Calculate the score of a possible assignment
# ---------------------------------------------------------

def calculate_assignment_score(
    patient,
    doctor,
    slot,
    current_date
):

    score = 0

    # Preferred doctor
    if (
        patient["preferred_doctor_id"] is not None
        and patient["preferred_doctor_id"] == doctor["doctor_id"]
    ):
        score += 40

    # Preferred date
    if slot["date"] == patient["preferred_date"]:
        score += 30

    # Preferred time
    if matches_time_preference(
        slot["start_time"],
        patient["preferred_time"]
    ):
        score += 20

    # Earlier appointment
    slot_date = datetime.strptime(
        slot["date"],
        "%Y-%m-%d"
    ).date()

    preference_date = datetime.strptime(
        patient["preferred_date"],
        "%Y-%m-%d"
    ).date()

    days_from_preference = abs(
        (slot_date - preference_date).days
    )

    # Maximum 10 points for the closest date
    score += max(
        0,
        10 - days_from_preference
    )

    return score


# ---------------------------------------------------------
# 3. Check whether an assignment satisfies the constraints
# ---------------------------------------------------------

def is_valid_assignment(
    patient,
    doctor,
    slot,
    schedule,
    doctor_daily_counts
):

    # Specialty compatibility
    if doctor["specialty"] != patient["required_specialty"]:
        return False

    # Slot must be available
    if slot["status"] != "Available":
        return False

    # Doctor daily capacity
    doctor_id = doctor["doctor_id"]
    slot_date = slot["date"]

    current_daily_count = doctor_daily_counts.get(
        (doctor_id, slot_date),
        0
    )

    if current_daily_count >= doctor["max_daily_appointments"]:
        return False

    # Patient cannot have another appointment
    # on the same date and time
    for existing in schedule:

        if (
            existing["patient_id"] == patient["patient_id"]
            and existing["date"] == slot_date
            and existing["start_time"] == slot["start_time"]
        ):
            return False

    return True

Scheduling Logic Result

The scheduling logic successfully defines the main decision process of the Single-Agent baseline.

The process separates feasibility from preference optimization: an assignment must first satisfy the scheduling constraints, and only then is it evaluated according to the patient’s preferences.

This makes the scheduling process interpretable and ensures that preference satisfaction does not override fundamental hospital constraints.

Run the Single-Agent Baseline

This section executes the Single-Agent scheduling process for all appointment requests.

Requests are processed sequentially. For each request, the scheduler searches for feasible appointments, evaluates the available candidates, selects the highest-scoring valid assignment, and reserves the corresponding time slot.

If no feasible appointment can be found, the request is recorded as unscheduled.

In [6]:
# Part 6 — Run Baseline

def run_baseline(
    appointment_requests_df,
    patients_df,
    doctors_df,
    slots_df
):

    # Create working copies
    working_slots = slots_df.copy()

    schedule = []

    doctor_daily_counts = {}

    unscheduled_requests = []

    # -----------------------------------------------------
    # Process each appointment request
    # -----------------------------------------------------

    for _, request in appointment_requests_df.iterrows():

        patient = patients_df[
            patients_df["patient_id"] == request["patient_id"]
        ].iloc[0]

        # -------------------------------------------------
        # Generate candidate assignments
        # -------------------------------------------------

        candidates = []

        for _, doctor in doctors_df.iterrows():

            # Quick specialty filtering
            if doctor["specialty"] != patient["required_specialty"]:
                continue

            doctor_slots = working_slots[
                working_slots["doctor_id"] == doctor["doctor_id"]
            ]

            for slot_index, slot in doctor_slots.iterrows():

                # Check constraints
                if not is_valid_assignment(
                    patient,
                    doctor,
                    slot,
                    schedule,
                    doctor_daily_counts
                ):
                    continue

                # Calculate score
                score = calculate_assignment_score(
                    patient,
                    doctor,
                    slot,
                    START_DATE
                )

                candidates.append({
                    "doctor_id": doctor["doctor_id"],
                    "slot_id": slot["slot_id"],
                    "date": slot["date"],
                    "start_time": slot["start_time"],
                    "end_time": slot["end_time"],
                    "score": score
                })

        # -------------------------------------------------
        # Select the best candidate
        # -------------------------------------------------

        if len(candidates) == 0:

            unscheduled_requests.append({
                "request_id": request["request_id"],
                "patient_id": request["patient_id"],
                "reason": "No valid assignment available"
            })

            continue

        # Sort candidates
        candidates = sorted(
            candidates,
            key=lambda x: (
                -x["score"],
                x["date"],
                x["start_time"]
            )
        )

        best_assignment = candidates[0]

        # -------------------------------------------------
        # Book the selected slot
        # -------------------------------------------------

        selected_slot_id = best_assignment["slot_id"]

        working_slots.loc[
            working_slots["slot_id"] == selected_slot_id,
            "status"
        ] = "Booked"

        # Update doctor's daily appointment count
        doctor_id = best_assignment["doctor_id"]
        appointment_date = best_assignment["date"]

        key = (doctor_id, appointment_date)

        doctor_daily_counts[key] = (
            doctor_daily_counts.get(key, 0) + 1
        )

        # -------------------------------------------------
        # Add assignment to final schedule
        # -------------------------------------------------

        schedule.append({
            "request_id": request["request_id"],
            "patient_id": patient["patient_id"],
            "doctor_id": best_assignment["doctor_id"],
            "required_specialty": patient["required_specialty"],
            "slot_id": best_assignment["slot_id"],
            "date": best_assignment["date"],
            "start_time": best_assignment["start_time"],
            "end_time": best_assignment["end_time"],
            "score": best_assignment["score"]
        })

    return (
        pd.DataFrame(schedule),
        pd.DataFrame(unscheduled_requests),working_slots
    )


# ---------------------------------------------------------
# Run the Baseline
# ---------------------------------------------------------

baseline_schedule_df, unscheduled_df, baseline_slots_df = run_baseline(
    appointment_requests_df,
    patients_df,
    doctors_df,
    slots_df
)


print("===== BASELINE RUN =====")
print("Total requests:", len(appointment_requests_df))
print("Scheduled requests:", len(baseline_schedule_df))
print("Unscheduled requests:", len(unscheduled_df))
print()

print("===== BASELINE SCHEDULE PREVIEW =====")
print(baseline_schedule_df.head(10))

===== BASELINE RUN =====
Total requests: 100
Scheduled requests: 100
Unscheduled requests: 0

===== BASELINE SCHEDULE PREVIEW =====
  request_id patient_id doctor_id  ... start_time end_time score
0       R001       P001      D019  ...      09:00    10:00   100
1       R002       P002      D005  ...      12:00    13:00    60
2       R003       P003      D010  ...      09:00    10:00   100
3       R004       P004      D004  ...      09:00    10:00    60
4       R005       P005      D012  ...      12:00    13:00   100
5       R006       P006      D003  ...      12:00    13:00   100
6       R007       P007      D002  ...      09:00    10:00    60
7       R008       P008      D014  ...      09:00    10:00   100
8       R009       P009      D019  ...      09:00    10:00    69
9       R010       P010      D004  ...      12:00    13:00    29

[10 rows x 9 columns]


Baseline Scheduling Result

The Single-Agent successfully scheduled all 100 appointment requests, with 0 unscheduled requests.

This represents a 100% successful assignment rate for the Easy experiment.

The result indicates that the available physician capacity and time slots were sufficient to satisfy all requests under the constraints used in this controlled experiment.

Final Baseline Schedule

This section organizes and displays the appointments generated by the Single-Agent scheduler.

The schedule is sorted by date, start time, and physician so that the assignments can be inspected in chronological order. The resulting table contains the patient, physician, specialty, appointment date and time, and the calculated assignment score.

Displaying the final schedule also allows the validity of individual assignments to be examined before the formal validation stage.

In [7]:
# Part 7 — Baseline Schedule

baseline_schedule_df = baseline_schedule_df.sort_values(
    by=["date", "start_time", "doctor_id"]
).reset_index(drop=True)


print("===== FINAL BASELINE SCHEDULE =====")
print("Total scheduled appointments:", len(baseline_schedule_df))
print()

print(
    baseline_schedule_df[
        [
            "request_id",
            "patient_id",
            "doctor_id",
            "slot_id",
            "required_specialty",
            "date",
            "start_time",
            "end_time",
            "score"
        ]
    ].to_string(index=False)
)

===== FINAL BASELINE SCHEDULE =====
Total scheduled appointments: 100

request_id patient_id doctor_id slot_id        required_specialty       date start_time end_time  score
      R040       P040      D001   S0001                Cardiology 2026-09-14      09:00    10:00     60
      R050       P050      D002   S0049               Dermatology 2026-09-14      09:00    10:00    100
      R004       P004      D004   S0145                Pediatrics 2026-09-14      09:00    10:00     60
      R078       P078      D007   S0289         Internal Medicine 2026-09-14      09:00    10:00     60
      R021       P021      D009   S0385             Ophthalmology 2026-09-14      09:00    10:00     60
      R003       P003      D010   S0433 Obstetrics and Gynecology 2026-09-14      09:00    10:00    100
      R096       P096      D011   S0481                Cardiology 2026-09-14      09:00    10:00     60
      R001       P001      D019   S0865             Ophthalmology 2026-09-14      09:00    10:00 

Final Schedule Result

The Single-Agent generated 100 scheduled appointments.

The schedule contains valid physician assignments and corresponding appointment times. The assignment score provides an indication of how well each selected appointment matches the patient’s preferences.

The schedule is now ready for formal constraint validation.

Baseline Schedule Validation

This section validates the generated schedule against the predefined scheduling constraints.

Validation is necessary because successfully generating a schedule does not automatically guarantee that all assignments are valid.

The validation process checks important constraints such as physician-specialty compatibility, slot availability, physician capacity, and scheduling conflicts.

Any detected violation is recorded so that the final schedule can be evaluated objectively.

In [8]:
# Part 8 — Baseline Validation

def validate_baseline_schedule(
    baseline_schedule_df,
    doctors_df,
    slots_df
):

    violations = []

    # ---------------------------------------------------------
    # 1. Check specialty compatibility
    # ---------------------------------------------------------

    for _, appointment in baseline_schedule_df.iterrows():

        doctor = doctors_df[
            doctors_df["doctor_id"] == appointment["doctor_id"]
        ].iloc[0]

        if doctor["specialty"] != appointment["required_specialty"]:

            violations.append({
                "patient_id": appointment["patient_id"],
                "violation": "Specialty mismatch"
            })

    # ---------------------------------------------------------
    # 2. Check duplicate slot assignments
    # ---------------------------------------------------------

    duplicate_slots = (
        baseline_schedule_df
        .groupby("slot_id")
        .size()
    )

    for slot_id, count in duplicate_slots.items():

        if count > 1:

            violations.append({
                "slot_id": slot_id,
                "violation": "Duplicate slot assignment"
            })

    # ---------------------------------------------------------
    # 3. Check patient overlapping appointments
    # ---------------------------------------------------------

    duplicate_patient_slots = (
        baseline_schedule_df
        .groupby(
            ["patient_id", "date", "start_time"]
        )
        .size()
    )

    for key, count in duplicate_patient_slots.items():

        if count > 1:

            violations.append({
                "patient_id": key[0],
                "date": key[1],
                "start_time": key[2],
                "violation": "Patient time conflict"
            })

    # ---------------------------------------------------------
    # 4. Check doctor daily capacity
    # ---------------------------------------------------------

    doctor_daily_counts = (
        baseline_schedule_df
        .groupby(["doctor_id", "date"])
        .size()
        .reset_index(name="appointment_count")
    )

    for _, row in doctor_daily_counts.iterrows():

        doctor = doctors_df[
            doctors_df["doctor_id"] == row["doctor_id"]
        ].iloc[0]

        if row["appointment_count"] > doctor["max_daily_appointments"]:

            violations.append({
                "doctor_id": row["doctor_id"],
                "date": row["date"],
                "violation": "Daily capacity exceeded"
            })

    # ---------------------------------------------------------
    # 5. Check that all assigned slots exist
    # ---------------------------------------------------------

    valid_slot_ids = set(slots_df["slot_id"])

    for _, appointment in baseline_schedule_df.iterrows():

        if appointment["slot_id"] not in valid_slot_ids:

            violations.append({
                "patient_id": appointment["patient_id"],
                "slot_id": appointment["slot_id"],
                "violation": "Invalid slot"
            })

    return pd.DataFrame(violations)


# ---------------------------------------------------------
# Run validation
# ---------------------------------------------------------

baseline_violations_df = validate_baseline_schedule(
    baseline_schedule_df,
    doctors_df,
    slots_df
)


print("===== BASELINE VALIDATION =====")
print("Total scheduled appointments:", len(baseline_schedule_df))
print("Total constraint violations:", len(baseline_violations_df))
print()

if len(baseline_violations_df) == 0:
    print("✓ Baseline schedule is valid.")
else:
    print(baseline_violations_df)

===== BASELINE VALIDATION =====
Total scheduled appointments: 100
Total constraint violations: 0

✓ Baseline schedule is valid.


Validation Result

The final schedule contains 100 scheduled appointments and 0 constraint violations.

This confirms that all generated assignments satisfy the constraints implemented in the validation process.

Therefore, the 100% assignment rate achieved by the baseline is not the result of accepting invalid assignments; all scheduled appointments are valid according to the defined rules.

Baseline Performance Evaluation

This section calculates the main performance metrics of the Single-Agent baseline.

The evaluation measures both scheduling effectiveness and patient preference satisfaction. The metrics include successful assignment rate, constraint violations, preferred physician satisfaction, preferred date satisfaction, preferred time satisfaction, and date deviation.

These metrics provide quantitative evidence that can later be used to compare the Single-Agent baseline with the Multi-Agent System.

In [9]:
# Part 9 — Baseline Evaluation

def calculate_baseline_metrics(
    baseline_schedule_df,
    appointment_requests_df,
    baseline_violations_df
):

    total_patients = len(appointment_requests_df)

    scheduled_patients = len(baseline_schedule_df)

    unscheduled_patients = (
        total_patients - scheduled_patients
    )

    # ---------------------------------------------------------
    # Successful Assignment Rate
    # ---------------------------------------------------------

    successful_assignment_rate = (
        scheduled_patients / total_patients
    ) * 100

    # ---------------------------------------------------------
    # Preferred Doctor Satisfaction
    # ---------------------------------------------------------

    requests_with_preferred_doctor = (
        appointment_requests_df[
            appointment_requests_df["preferred_doctor_id"].notna()
        ]
    )

    if len(requests_with_preferred_doctor) > 0:

        doctor_check = baseline_schedule_df.merge(
            requests_with_preferred_doctor[
                ["request_id", "preferred_doctor_id"]
            ],
            on="request_id",
            how="inner"
        )

        preferred_doctor_satisfaction = (
            (
                doctor_check["doctor_id"]
                == doctor_check["preferred_doctor_id"]
            ).mean()
            * 100
        )

    else:
        preferred_doctor_satisfaction = 0

    # ---------------------------------------------------------
    # Preferred Date Satisfaction
    # ---------------------------------------------------------

    date_check = baseline_schedule_df.merge(
        appointment_requests_df[
            ["request_id", "preferred_date"]
        ],
        on="request_id",
        how="left"
    )

    preferred_date_satisfaction = (
        (
            date_check["date"]
            == date_check["preferred_date"]
        ).mean()
        * 100
    )

    # ---------------------------------------------------------
    # Preferred Time Satisfaction
    # ---------------------------------------------------------

    time_check = baseline_schedule_df.merge(
        appointment_requests_df[
            ["request_id", "preferred_time"]
        ],
        on="request_id",
        how="left"
    )

    def check_time(row):

        if row["preferred_time"] == "No Preference":
            return None

        return matches_time_preference(
            row["start_time"],
            row["preferred_time"]
        )

    time_results = time_check.apply(
        check_time,
        axis=1
    )

    preferred_time_satisfaction = (
        time_results.dropna().mean()
        * 100
    )

    # ---------------------------------------------------------
    # Date Deviation
    # ---------------------------------------------------------

    date_check["date_deviation"] = (
        pd.to_datetime(date_check["date"])
        - pd.to_datetime(date_check["preferred_date"])
    ).dt.days.abs()

    average_date_deviation = (
        date_check["date_deviation"].mean()
    )

    maximum_date_deviation = (
        date_check["date_deviation"].max()
    )

    # ---------------------------------------------------------
    # Constraint Violations
    # ---------------------------------------------------------

    constraint_violations = len(
        baseline_violations_df
    )

    # ---------------------------------------------------------
    # Create metrics dictionary
    # ---------------------------------------------------------

    metrics = {
        "Total Patients": total_patients,
        "Scheduled Patients": scheduled_patients,
        "Unscheduled Patients": unscheduled_patients,
        "Successful Assignment Rate (%)":
            successful_assignment_rate,
        "Constraint Violations":
            constraint_violations,
        "Preferred Doctor Satisfaction (%)":
            preferred_doctor_satisfaction,
        "Preferred Date Satisfaction (%)":
            preferred_date_satisfaction,
        "Preferred Time Satisfaction (%)":
        preferred_time_satisfaction,
        "Average Date Deviation (days)":
            average_date_deviation,
        "Maximum Date Deviation (days)":
            maximum_date_deviation
    }

    return metrics


# ---------------------------------------------------------
# Calculate Baseline Metrics
# ---------------------------------------------------------

baseline_metrics = calculate_baseline_metrics(
    baseline_schedule_df,
    appointment_requests_df,
    baseline_violations_df
)


print("===== BASELINE EVALUATION =====")

for metric, value in baseline_metrics.items():

    if isinstance(value, float):
        print(f"{metric}: {value:.2f}")
    else:
        print(f"{metric}: {value}")

===== BASELINE EVALUATION =====
Total Patients: 100
Scheduled Patients: 100
Unscheduled Patients: 0
Successful Assignment Rate (%): 100.00
Constraint Violations: 0
Preferred Doctor Satisfaction (%): 95.24
Preferred Date Satisfaction (%): 76.00
Preferred Time Satisfaction (%): 100.00
Average Date Deviation (days): 0.27
Maximum Date Deviation (days): 2


Baseline Evaluation Result

The Easy Single-Agent baseline achieved the following results:

• Successful Assignment Rate: 100.00%
• Constraint Violations: 0
• Preferred Doctor Satisfaction: 95.24%
• Preferred Date Satisfaction: 76.00%
• Preferred Time Satisfaction: 100.00%
• Average Date Deviation: 0.27 days
• Maximum Date Deviation: 2 days

The results show that the baseline was able to schedule every request without violating the defined constraints while achieving high satisfaction of patient preferences.

This experiment therefore establishes a successful initial baseline before increasing the dataset size and introducing dynamic scheduling events.

Save Baseline Results

This section saves the generated baseline outputs as separate CSV files.

Saving the results allows the final schedule, unscheduled requests, constraint violations, and evaluation metrics to be reused later without rerunning the entire experiment.

These saved outputs can also support the later comparison between the Single-Agent and Multi-Agent systems.

In [10]:
# Part 10 — Save Baseline Results

# ---------------------------------------------------------
# 1. Save the final schedule
# ---------------------------------------------------------

baseline_schedule_df.to_csv(
    "baseline_schedule.csv",
    index=False
)

# ---------------------------------------------------------
# 2. Save unscheduled requests
# ---------------------------------------------------------

unscheduled_df.to_csv(
    "baseline_unscheduled.csv",
    index=False
)

# ---------------------------------------------------------
# 3. Save validation violations
# ---------------------------------------------------------

baseline_violations_df.to_csv(
    "baseline_violations.csv",
    index=False
)

# ---------------------------------------------------------
# 4. Save evaluation metrics
# ---------------------------------------------------------

baseline_metrics_df = pd.DataFrame(
    list(baseline_metrics.items()),
    columns=["Metric", "Value"]
)

baseline_metrics_df.to_csv(
    "baseline_metrics.csv",
    index=False
)


print("===== BASELINE RESULTS SAVED =====")
print("✓ baseline_schedule.csv")
print("✓ baseline_unscheduled.csv")
print("✓ baseline_violations.csv")
print("✓ baseline_metrics.csv")

===== BASELINE RESULTS SAVED =====
✓ baseline_schedule.csv
✓ baseline_unscheduled.csv
✓ baseline_violations.csv
✓ baseline_metrics.csv


Saved Results
The baseline results were successfully saved as:
baseline_schedule.csv
baseline_unscheduled.csv
baseline_violations.csv
baseline_metrics.csv
These files preserve the outputs of the Easy Single-Agent experiment for later analysis and comparison.

## M4 Live Demo Verification — Easy / Static Baseline

This verification block is part of the full project notebook and is also copied **unchanged** into the simplified M4 demo notebook.
It verifies the same Easy / Static baseline used in the live 5-minute demonstration.


In [11]:
# M4 — End-to-End Baseline Verification
m4_total_requests = len(appointment_requests_df)
m4_scheduled = len(baseline_schedule_df)
m4_unscheduled = len(unscheduled_df)
m4_violations = len(baseline_violations_df)
m4_assignment_rate = (m4_scheduled / m4_total_requests * 100) if m4_total_requests else 0

m4_end_to_end_success = (
    m4_total_requests == 100
    and m4_scheduled == 100
    and m4_unscheduled == 0
    and m4_violations == 0
)

print("===== M4 END-TO-END VERIFICATION =====")
print(f"End-to-end task completed: {m4_end_to_end_success}")
print(f"Total requests: {m4_total_requests}")
print(f"Scheduled: {m4_scheduled}")
print(f"Unscheduled: {m4_unscheduled}")
print(f"Constraint violations: {m4_violations}")
print(f"Assignment rate: {m4_assignment_rate:.2f}%")
assert m4_end_to_end_success, "M4 baseline verification failed."
print("Baseline valid: True")


===== M4 END-TO-END VERIFICATION =====
End-to-end task completed: True
Total requests: 100
Scheduled: 100
Unscheduled: 0
Constraint violations: 0
Assignment rate: 100.00%
Baseline valid: True


## M4 Demo Scope

This notebook intentionally demonstrates the Easy / Static Single-Agent baseline end to end within the 5-minute live demo. The full project notebook retains Experiment 2 for the complete research evaluation.
